In [2]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from trajectory_plotter import TrajectoryValidationRiskPloter

In [1]:
print ("hey")

hey


In [ ]:
# --- plotter instance (recreated on each run via buttons) ---
plotter = TrajectoryValidationRiskPloter(random_seed=42)

In [ ]:
# --- Build figure with 5 traces ---
fig = go.FigureWidget()
fig.add_scatter(x=[], y=[], name='Train',        line=dict(color='steelblue'))
fig.add_scatter(x=[], y=[], name='Valid',        line=dict(color='tomato'))
fig.add_scatter(x=[], y=[], name='Holdout Test', line=dict(color='seagreen'))
fig.add_scatter(x=[], y=[], name='LOOCV',        line=dict(color='mediumpurple'))
fig.add_scatter(x=[], y=[], name='LOOCV Test',   line=dict(color='darkorange'))
fig.update_layout(
    xaxis_title='Iteration t', yaxis_title='MSE',
    title='GD Trajectory', height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
)

# --- Scale toggles (green = selected) ---
def make_toggle_pair(label):
    b_lin = widgets.ToggleButton(value=True,  description='linear',
                                  button_style='success', layout=widgets.Layout(width='70px', height='28px'))
    b_log = widgets.ToggleButton(value=False, description='log',
                                  button_style='',        layout=widgets.Layout(width='70px', height='28px'))
    def on_lin(change):
        if change['new']:
            b_log.value = False; b_log.button_style = ''; b_lin.button_style = 'success'
        elif not b_log.value:
            b_lin.value = True
        update_scales()
    def on_log(change):
        if change['new']:
            b_lin.value = False; b_lin.button_style = ''; b_log.button_style = 'success'
        elif not b_lin.value:
            b_log.value = True
        update_scales()
    b_lin.observe(on_lin, names='value')
    b_log.observe(on_log, names='value')
    row = widgets.HBox([widgets.Label(label, layout=widgets.Layout(width='50px')), b_lin, b_log])
    return row, b_lin, b_log

x_row, x_lin, x_log = make_toggle_pair('X scale:')
y_row, y_lin, y_log = make_toggle_pair('Y scale:')

def update_scales():
    fig.update_layout(xaxis_type='log' if x_log.value else 'linear',
                      yaxis_type='log' if y_log.value else 'linear')

# --- Parameter inputs ---
def parse_int(s):
    s = str(s).strip().lower()
    if s.endswith('k'):
        return int(float(s[:-1]) * 1000)
    return int(float(s))

st = {'description_width': '42px'}
lo = widgets.Layout(width='118px')
n_input    = widgets.Text(value='200',  description='n:',     style=st, layout=lo)
p_input    = widgets.Text(value='400',  description='p:',     style=st, layout=lo)
seed_input = widgets.Text(value='42',   description='seed:',  style=st, layout=lo)
eta_input  = widgets.Text(value='0.1',  description='eta:',   style=st, layout=lo)
maxT_input = widgets.Text(value='500',  description='max_T:', style=st, layout=lo)
m_input    = widgets.Text(value='200',  description='m:',     style=st, layout=lo)
ntest_input= widgets.Text(value='1000', description='n_test:',style=st, layout=lo)

holdout_test_cb = widgets.Checkbox(value=False, description='Track Holdout Test', indent=False)
loocv_test_cb   = widgets.Checkbox(value=False, description='Track LOOCV Test',   indent=False)

holdout_btn = widgets.Button(description='Run Holdout', button_style='primary', layout=widgets.Layout(width='110px'))
loocv_btn   = widgets.Button(description='Run LOOCV',   button_style='warning',  layout=widgets.Layout(width='110px'))
status      = widgets.Label(value='')

def on_holdout(b):
    status.value = 'Running Holdout...'
    holdout_btn.disabled = True; loocv_btn.disabled = True
    try:
        p = TrajectoryValidationRiskPloter(random_seed=parse_int(seed_input.value))
        track_test = holdout_test_cb.value
        train, valid, test = p.run_hold_out_GD(
            n=parse_int(n_input.value), p=parse_int(p_input.value),
            m=parse_int(m_input.value), max_T=parse_int(maxT_input.value),
            eta=float(eta_input.value), test_error_tracking=track_test,
            test_sample_size=parse_int(ntest_input.value),
        )
        ts = list(range(len(train)))
        with fig.batch_update():
            fig.data[0].x = ts; fig.data[0].y = train
            fig.data[1].x = ts; fig.data[1].y = valid
            fig.data[2].x = ts; fig.data[2].y = test if track_test else []
        status.value = f'Holdout done. ({len(train)} iters)'
    except Exception as e:
        status.value = f'Error: {e}'
    holdout_btn.disabled = False; loocv_btn.disabled = False

def on_loocv(b):
    status.value = 'Running LOOCV... (slow)'
    holdout_btn.disabled = True; loocv_btn.disabled = True
    try:
        p = TrajectoryValidationRiskPloter(random_seed=parse_int(seed_input.value))
        track_test = loocv_test_cb.value
        loocv, loocv_test = p.generate_samples_and_run_LOOCV(
            n=parse_int(n_input.value), p=parse_int(p_input.value),
            max_T=parse_int(maxT_input.value), eta=float(eta_input.value),
            test_error_tracking=track_test,
            test_sample_size=parse_int(ntest_input.value),
        )
        ts = list(range(len(loocv)))
        with fig.batch_update():
            fig.data[3].x = ts; fig.data[3].y = loocv
            fig.data[4].x = ts; fig.data[4].y = loocv_test if track_test else []
        status.value = f'LOOCV done. ({len(loocv)} iters)'
    except Exception as e:
        status.value = f'Error: {e}'
    holdout_btn.disabled = False; loocv_btn.disabled = False

holdout_btn.on_click(on_holdout)
loocv_btn.on_click(on_loocv)

# --- Layout ---
display(widgets.VBox([
    widgets.HBox([x_row, y_row]),
    fig,
    widgets.HTML('<hr style="margin:6px 0">'),
    widgets.HBox([n_input, p_input, seed_input, eta_input, maxT_input, m_input, ntest_input]),
    widgets.HBox([holdout_btn, holdout_test_cb]),
    widgets.HBox([loocv_btn,   loocv_test_cb]),
    widgets.HBox([status]),
]))